# YouTube Revenge Factory - 4K Video Renderer

This notebook renders high-quality 4K videos from story content, audio, and background images. It creates professional YouTube-style videos with Ken Burns effects, subtitles, and audio overlays.

## Features
- Mount Google Drive for file storage
- Download assets from GitHub Actions via gdown
- Render 4K (3840x2160) videos with Ken Burns effect
- Add TTS audio and burned-in subtitles
- Export as video.mp4
- Upload results back to Google Drive


In [ ]:
import json
import os
import subprocess
import tempfile
from pathlib import Path
import gdown
import ffmpeg
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import shutil

# Configuration
VIDEO_WIDTH = 3840
VIDEO_HEIGHT = 2160
FPS = 30
SCENE_DURATION = 5  # seconds per scene

# Set up paths
drive_path = '/content/drive'
if not os.path.exists(drive_path):
    from google.colab import drive
    drive.mount(drive_path)

# Create working directories
working_dir = Path('/content/working')
working_dir.mkdir(exist_ok=True)

assets_dir = working_dir / 'assets'
assets_dir.mkdir(exist_ok=True)

output_dir = working_dir / 'output'
output_dir.mkdir(exist_ok=True)


## Download Assets from GitHub Actions

Download story JSON, audio files, and background images from GitHub Actions using gdown.


In [ ]:
def download_assets():
    """Download assets from GitHub Actions."""
    try:
        # Download story JSON
        print('Downloading story JSON...')
        gdown.download('https://github.com/user/repo/releases/download/assets/story.json',
                       str(assets_dir / 'story.json'), quiet=False)
        
        # Download audio files
        print('Downloading audio files...')
        for i in range(1, 6):  # Assuming 5 audio files
            gdown.download(f'https://github.com/user/repo/releases/download/assets/audio_{i}.wav',
                           str(assets_dir / f'audio_{i}.wav'), quiet=False)
        
        # Download background images
        print('Downloading background images...')
        for i in range(1, 6):  # Assuming 5 background images
            gdown.download(f'https://github.com/user/repo/releases/download/assets/background_{i}.jpg',
                           str(assets_dir / f'background_{i}.jpg'), quiet=False)
        
        print('All assets downloaded successfully!')
    except Exception as e:
        print(f'Error downloading assets: {e}')
        raise

# Execute download
download_assets()


## Load Story Data and Setup Video Composition

Load story JSON and prepare for video rendering with Ken Burns effects.


In [ ]:
def load_story_data():
    """Load story JSON data."""
    with open(assets_dir / 'story.json', 'r', encoding='utf-8') as f:
        story = json.load(f)
    return story

def create_ken_burns_frames(image_path, duration, start_zoom=1.0, end_zoom=1.2):
    """Create Ken Burns effect frames for an image.

    Args:
        image_path: Path to the background image
        duration: Duration in seconds
        start_zoom: Starting zoom level
        end_zoom: Ending zoom level

    Returns:
        List of frame paths
    """
    frames_dir = working_dir / 'frames'
    frames_dir.mkdir(exist_ok=True)

    # Load and resize image
    img = Image.open(image_path)
    img = img.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)

    frames = []
    for frame_idx in range(duration * FPS):
        # Calculate zoom level for this frame
        t = frame_idx / (duration * FPS)
        zoom = start_zoom + (end_zoom - start_zoom) * t

        # Apply Ken Burns effect
        new_width = int(VIDEO_WIDTH / zoom)
        new_height = int(VIDEO_HEIGHT / zoom)

        # Crop center of image
        left = (VIDEO_WIDTH - new_width) // 2
        top = (VIDEO_HEIGHT - new_height) // 2
        right = left + new_width
        bottom = top + new_height

        cropped = img.crop((left, top, right, bottom))

        # Resize back to 4K
        frame_img = cropped.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)

        # Save frame
        frame_path = frames_dir / f'frame_{len(frames):04d}.png'
        frame_img.save(frame_path, 'PNG')
        frames.append(str(frame_path))

    return frames

def create_subtitles(story, scene_idx):
    """Create subtitles for a specific scene.

    Args:
        story: Story data
        scene_idx: Scene index (0-based)

    Returns:
        Subtitle text for the scene
    """
    if scene_idx < len(story['scenes']):
        return story['scenes'][scene_idx]['text']
    return f"Scene {scene_idx + 1}"

def add_subtitles_to_frame(frame_path, subtitle_text, font_size=48):
    """Add burned-in subtitles to a video frame.

    Args:
        frame_path: Path to the frame image
        subtitle_text: Subtitle text to add
        font_size: Font size for subtitles

    Returns:
        Path to the frame with subtitles
    """
    # Load frame
    frame = Image.open(frame_path)
    draw = ImageDraw.Draw(frame)

    # Try to load a font (use default if not available)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', font_size)
    except:
        font = ImageFont.load_default()

    # Calculate text position (bottom of frame)
    bbox = draw.textbbox((0, 0), subtitle_text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    text_x = (VIDEO_WIDTH - text_width) // 2
    text_y = VIDEO_HEIGHT - text_height - 50  # 50px from bottom

    # Add text shadow for better visibility
    shadow_offset = 2
    draw.text((text_x + shadow_offset, text_y + shadow_offset), subtitle_text, fill='black', font=font)
    draw.text((text_x, text_y), subtitle_text, fill='white', font=font)

    # Save frame with subtitles
    output_path = working_dir / 'frames' / f'sub_{Path(frame_path).name}'
    frame.save(output_path, 'PNG')
    return str(output_path)

# Load story data
story = load_story_data()
print(f'Story loaded: {story["title"]}')

# Create frames directory
frames_dir = working_dir / 'frames'
frames_dir.mkdir(exist_ok=True, parents=True)


## Generate Video Frames with Ken Burns Effect

Generate video frames for each scene with Ken Burns zoom effects and subtitles.


In [ ]:
def generate_video_frames():
    """Generate all video frames for the story."""
    all_frames = []

    for scene_idx in range(len(story['scenes'])):
        scene = story['scenes'][scene_idx]
        print(f'Processing scene {scene_idx + 1}: {scene["title"]}')

        # Download background image
        bg_path = assets_dir / f'background_{scene_idx + 1}.jpg'
        if not bg_path.exists():
            print(f'Warning: Background image {bg_path} not found')
            continue

        # Create Ken Burns frames
        scene_frames = create_ken_burns_frames(bg_path, SCENE_DURATION)

        # Add subtitles to each frame
        subtitle_text = create_subtitles(story, scene_idx)
        subtitled_frames = []
        for frame_path in scene_frames:
            subtitled_frame = add_subtitles_to_frame(frame_path, subtitle_text)
            subtitled_frames.append(subtitled_frame)

        all_frames.extend(subtitled_frames)
        print(f'  Generated {len(subtitled_frames)} frames with subtitles')

    print(f'Total frames generated: {len(all_frames)}')
    return all_frames

# Generate all frames
video_frames = generate_video_frames()


## Combine Frames into Video with Audio

Combine all frames into a video and add TTS audio overlay.


In [ ]:
def combine_frames_to_video(frames):
    """Combine frames into a video file.

    Args:
        frames: List of frame image paths

    Returns:
        Path to the output video
    """
    # Sort frames by name
    frames.sort(key=lambda x: int(Path(x).stem.split('_')[1]))

    # Create input file for ffmpeg
    input_file = working_dir / 'input.txt'
    with open(input_file, 'w') as f:
        for frame in frames:
            f.write(f'file "{frame}"\n')

    # Use ffmpeg to create video
    video_path = output_dir / 'video.mp4'
    cmd = [
        'ffmpeg', '-y', '-f', 'concat', '-i', str(input_file),
        '-c:v', 'libx264', '-preset', 'fast', '-b:v', '5000k',
        '-pix_fmt', 'yuv420p', '-r', str(FPS), str(video_path)
    ]

    print('Creating video from frames...')
    subprocess.run(cmd, check=True)

    print(f'Video created: {video_path}')
    return str(video_path)

def add_audio_to_video(video_path, audio_path):
    """Add audio to video file.

    Args:
        video_path: Path to the video file
        audio_path: Path to the audio file

    Returns:
        Path to the video with audio
    """
    # Create output path
    output_path = working_dir / 'video_with_audio.mp4'

    # Use ffmpeg to add audio
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path), '-i', str(audio_path),
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', str(output_path)
    ]

    print('Adding audio to video...')
    subprocess.run(cmd, check=True)

    print(f'Video with audio created: {output_path}')
    return str(output_path)

# Combine frames into video
video_file = combine_frames_to_video(video_frames)

# Add audio (using first audio file as example)
audio_file = assets_dir / 'audio_1.wav'
if audio_file.exists():
    final_video = add_audio_to_video(video_file, audio_file)
else:
    print('Warning: Audio file not found, using video without audio')
    final_video = video_file


## Upload Results to Google Drive

Upload the final video and any additional assets back to Google Drive.


In [ ]:
def upload_to_drive():
    """Upload results to Google Drive."""
    try:
        # Upload final video
        print('Uploading final video to Google Drive...')
        gdown.download_folder(str(output_dir),
                               output=str(drive_path) / 'output',
                               quiet=False)

        # Upload story JSON
        print('Uploading story JSON to Google Drive...')
        gdown.download(str(assets_dir / 'story.json'),
                       str(drive_path) / 'story.json', quiet=False)

        print('All files uploaded to Google Drive successfully!')
    except Exception as e:
        print(f'Error uploading to Google Drive: {e}')
        raise

# Upload results
upload_to_drive()

print('=== Video rendering complete! ===')
print(f'Final video location: {drive_path}/output/video.mp4')
